# Raw binary file inspector

Open **one raw Slocum binary file** (`.dbd`/`.ebd`/`.sbd`/`.tbd`/`.mbd`/`.nbd`)
directly with `dbdreader`, no mission config or `deployment.yml` involved --
for when you need to know exactly what's *in* a specific file (metadata,
every declared variable, how much of it is actually valid data), or to
confirm a file is genuinely corrupt rather than guess from a traceback.

`open_binary()` below is the reusable piece: point it at any file + the
master cache, get back a ready-to-query `dbdreader.DBD` object. Re-run the
"pick a file" cell with a different path to check another one.


## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import dbdreader
from dbdreader.dbdreader import DBDHeader, DbdError

from norgliders_data_pipeline.settings import load_settings

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)


## 1. `open_binary()` -- the reusable function

Wraps `dbdreader.DBD()` with clear diagnostics for the two failure modes
actually seen in practice (see mission 12/14's processing notes):

- the referenced `.cac` cache file isn't in `cache_dir`
- the file's own inline sensor list (`sensor_list_factored: 0`) is
  **truncated** -- declares more sensors than it actually contains,
  because the file itself is an incomplete telemetry transmission or a
  cut-off transfer. `dbdreader` surfaces this as a bare `IndexError`
  deep in its cache parser; this re-diagnoses it properly by counting
  the file's actual lines against what its header claims.


In [ ]:
def open_binary(path: str | Path, cache_dir: str | Path | None = None) -> dbdreader.DBD:
    """Open one raw Slocum binary file with dbdreader, cache resolved
    against cache_dir (default: the master cache from processing.toml).

    Raises RuntimeError with a specific diagnosis instead of dbdreader's
    bare/confusing exceptions, for the two failure modes seen in practice.
    """
    path = Path(path)
    cache_dir = Path(cache_dir) if cache_dir else load_settings().master_cache_dir
    if not path.is_file():
        raise FileNotFoundError(f"no such file: {path}")

    try:
        return dbdreader.DBD(str(path), cacheDir=str(cache_dir))
    except IndexError as exc:
        # Inline sensor list (factored=0) cut off partway -- count how far
        # it actually got vs. what the header declares.
        h = DBDHeader()
        with open(path, "rb") as fp:
            h.read_header(fp)
            n = sum(1 for _ in iter(fp.readline, b""))
        declared = h.info.get("total_num_sensors")
        raise RuntimeError(
            f"{path.name}: inline sensor list truncated -- header declares "
            f"total_num_sensors={declared}, file only has {n} lines after "
            f"the header. The file itself is incomplete at the source "
            f"(bad telemetry transfer / truncated copy), not something "
            f"cache_dir or re-running fixes."
        ) from exc
    except DbdError as exc:
        raise RuntimeError(f"{path.name}: {exc}") from exc


## 2. Pick a file

In [ ]:
FILE = Path("/Data/gfi/projects/slocum/data/delayed/012-freyja_naco_porsangerfjorden_jun2014"
            "/binary/freyja-2014-158-6-1003.sbd")
CACHE_DIR = load_settings().master_cache_dir   # override if you want a mission-local cache/ instead

d = open_binary(FILE, CACHE_DIR)
print(f"opened {FILE.name}  ({FILE.stat().st_size:,} bytes)")


## 3. Metadata (the file's own ASCII header)

In [ ]:
meta = {k: v for k, v in d.headerInfo.items() if k != "parameter_list"}
meta["cache_found"] = d.cacheFound
meta["cache_id"] = d.cacheID
pd.Series(meta, name="value").to_frame()


## 4. Variables -- size, units, and how much is actually valid data

`total_num_sensors` above is the *catalogue* this file's cache references
(everything the glider could ever log under this sensor-list config) --
`sensors_per_cycle` / the table below is what *this file* actually carries.
`n_valid` counts finite (non-NaN) samples per variable, same check that
caught mission 002's FLNTU being aboard-but-silent and mission 12's
missing oxygen channel.


In [ ]:
rows = []
for name, size in zip(d.parameterNames, d.byteSizes):
    t, v = d.get(name)
    ok = np.isfinite(v)
    rows.append({
        "variable": name,
        "size_bytes": size,
        "unit": d.parameterUnits.get(name, ""),
        "n_samples": len(v),
        "n_valid": int(ok.sum()),
        "pct_valid": round(100 * ok.sum() / len(v), 1) if len(v) else 0.0,
        "min": np.nanmin(v) if ok.any() else np.nan,
        "max": np.nanmax(v) if ok.any() else np.nan,
    })
variables = pd.DataFrame(rows).set_index("variable").sort_index()
print(f"{len(variables)} variable(s) in this file "
      f"(of {d.headerInfo['total_num_sensors']} in the sensor-list catalogue)")
variables


## 5. Check another file

Close this one, then re-open with a different path -- `open_binary()` is
the whole point of keeping this reusable. Re-run cells 3/4 against `d`
afterwards, or loop over several files to compare (e.g. `pd.concat` the
`variables` tables with a `source` column) if you want more than one at once.


In [ ]:
d.close()

FILE = Path("/Data/gfi/projects/slocum/data/delayed/012-freyja_naco_porsangerfjorden_jun2014"
            "/binary/freyja-2014-158-6-1012.tbd")   # edit to point anywhere -- .ebd/.dbd/.tbd/.sbd all work
d = open_binary(FILE, CACHE_DIR)
print(f"opened {FILE.name}  ({FILE.stat().st_size:,} bytes)")


## Appendix: what a corrupt file looks like here

`open_binary()` on a truncated file (declares more inline sensors than it
actually has -- the failure mode behind mission 12/14's `-0` segments)
raises a clear `RuntimeError` instead of `dbdreader`'s bare `IndexError`:

```
RuntimeError: freyja-2014-156-2-0.sbd: inline sensor list truncated --
header declares total_num_sensors=1879, file only has 217 lines after
the header. The file itself is incomplete at the source (bad telemetry
transfer / truncated copy), not something cache_dir or re-running fixes.
```
